In [43]:
import numpy as np
import pandas as pd
import joblib
import re
from urllib.parse import urlparse

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

In [44]:
url_model = joblib.load("../model/url_model_calibrated.pkl")
sms_model = joblib.load("../model/sms_model_calibrated.pkl")
vectorizer = joblib.load("../model/tfidf_vectorizer.pkl")

In [45]:
url_df = pd.read_csv("../data/malicious_phish.csv")

sms_df = pd.read_csv("../data/spam.csv", encoding='latin-1')
sms_df = sms_df[['v1','v2']]
sms_df.columns = ['label','message']

In [34]:
sms_df = pd.read_csv("../data/spam.csv", encoding='latin-1')
sms_df = sms_df[['v1', 'v2']]
sms_df.columns = ['label', 'message']
sms_df['label'] = sms_df['label'].map({'ham': 0, 'spam': 1})

In [46]:
def extract_url_features(url):
    parsed = urlparse(url)

    return [
        len(url),
        url.count('.'),
        url.count('-'),
        url.count('@'),
        url.count('?'),
        url.count('='),
        url.count('/'),
        url.count('www'),
        1 if parsed.scheme == 'https' else 0,
        1 if re.match(r"^\d+\.\d+\.\d+\.\d+", parsed.netloc) else 0,
        1 if any(k in url.lower() for k in ["login","verify","bank","secure","account"]) else 0,
        len(parsed.netloc)
    ]

In [47]:
url_features = url_df['url'].apply(lambda x: extract_url_features(x))
url_features = pd.DataFrame(url_features.tolist())

In [48]:
url_df['label'] = url_df['type'].apply(lambda x: 0 if x == 'benign' else 1)
url_labels = url_df['label'].values

In [49]:
sms_features = vectorizer.transform(sms_df['message'])
sms_labels = sms_df['label'].map({'ham':0, 'spam':1}).values

In [50]:
url_probs = url_model.predict_proba(url_features)[:,1]
sms_probs = sms_model.predict_proba(sms_features)[:,1]

In [51]:
n = min(len(url_probs), len(sms_probs))

url_probs = url_probs[:n]
sms_probs = sms_probs[:n]
labels = url_labels[:n]

In [52]:
from sklearn.linear_model import LogisticRegression

meta_model = LogisticRegression()
meta_model.fit(fusion_X, fusion_y)

,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary ` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :term:`mul

In [53]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

def evaluate_model(model, X, y):
    y_pred = model.predict(X)
    y_prob = model.predict_proba(X)[:,1]
    
    print("Accuracy:", accuracy_score(y, y_pred))
    print("Precision:", precision_score(y, y_pred))
    print("Recall:", recall_score(y, y_pred))
    print("F1 Score:", f1_score(y, y_pred))
    print("ROC AUC:", roc_auc_score(y, y_prob))
    print("-"*40)

evaluate_model(meta_model, fusion_X, fusion_y)

Accuracy: 0.9560301507537688
Precision: 0.9473304473304474
Recall: 0.8841750841750842
F1 Score: 0.9146638801811215
ROC AUC: 0.987414723039876
----------------------------------------


In [54]:
joblib.dump(meta_model, "../model/fusion_model.pkl")

['../model/fusion_model.pkl']

In [55]:
url_features.shape[1]

12

In [39]:
sms_probs = sms_model.predict_proba(sms_features)[:,1]

In [40]:
len(url_probs) == len(sms_probs)

True

In [41]:
print(len(url_probs), len(sms_probs))

5572 5572


In [42]:
fusion_X = np.vstack((url_probs, sms_probs)).T
fusion_y = y

NameError: name 'y' is not defined

In [38]:
url_probs = url_model.predict_proba(url_features)[:,1]
sms_probs = sms_model.predict_proba(sms_features)[:,1]

ValueError: Feature shape mismatch, expected: 12, got 10

In [ ]:
fusion_X = np.vstack((url_probs, sms_probs)).T
fusion_y = labels

In [ ]:
from sklearn.linear_model import LogisticRegression

meta_model = LogisticRegression()
meta_model.fit(fusion_X, fusion_y)

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

def evaluate_model(model, X, y):
    y_pred = model.predict(X)
    y_prob = model.predict_proba(X)[:,1]
    
    print("Accuracy:", accuracy_score(y, y_pred))
    print("Precision:", precision_score(y, y_pred))
    print("Recall:", recall_score(y, y_pred))
    print("F1 Score:", f1_score(y, y_pred))
    print("ROC AUC:", roc_auc_score(y, y_prob))
    print("-"*40)

In [ ]:
print("Fusion Model Results")
evaluate_model(meta_model, fusion_X, fusion_y)

In [ ]:
from sklearn.model_selection import train_test_split

X_train_f, X_test_f, y_train_f, y_test_f = train_test_split(
    fusion_X, fusion_y, test_size=0.2, random_state=42, stratify=fusion_y
)

In [ ]:
meta_model = LogisticRegression()
meta_model.fit(X_train_f, y_train_f)

In [ ]:
print("Fusion Model (Proper Evaluation)")
evaluate_model(meta_model, X_test_f, y_test_f)

In [ ]:
fusion_weighted = 0.6 * url_probs + 0.4 * sms_probs

In [ ]:
from sklearn.metrics import roc_auc_score

print("ROC-AUC:", roc_auc_score(fusion_y, fusion_weighted))

In [ ]:
import numpy as np

thresholds = np.arange(0.1, 0.9, 0.1)

for t in thresholds:
    y_pred = (meta_model.predict_proba(X_test_f)[:,1] >= t).astype(int)
    
    from sklearn.metrics import recall_score, precision_score, f1_score
    
    print(f"Threshold: {t}")
    print("Precision:", precision_score(y_test_f, y_pred))
    print("Recall:", recall_score(y_test_f, y_pred))
    print("F1:", f1_score(y_test_f, y_pred))
    print("-"*30)

In [ ]:
joblib.dump(meta_model, "../model/fusion_model.pkl")

In [ ]:
len(extract_url_features("http://test.com"))